<a href="https://colab.research.google.com/github/SaliElloh/deepfake-research-rag/blob/main/finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# finetuning file
import json
import os
# from pypdf import PdfReader
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# import ollama


In [2]:
!nvidia-smi


Sun Sep  6 16:57:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!git clone https://github.com/SaliElloh/deepfake-research-rag.git


fatal: destination path 'deepfake-research-rag' already exists and is not an empty directory.


In [4]:
!pip install unsloth

FileNotFoundError: [Errno 2] No such file or directory: 'deepfake-research-rag/training_data.jsonl'

In [9]:
!rm -rf deepfake-research-rag
!git clone https://github.com/SaliElloh/deepfake-research-rag.git
!ls deepfake-research-rag/

Cloning into 'deepfake-research-rag'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 60 (delta 22), reused 43 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 8.67 MiB | 32.75 MiB/s, done.
Resolving deltas: 100% (22/22), done.
app.py		corpus		     __pycache__  training_data_clean.jsonl
build_index.py	finetune.ipynb	     query.py
chroma_db	generate_dataset.py  README


In [10]:
!ls deepfake-research-rag/

app.py		corpus		     __pycache__  training_data_clean.jsonl
build_index.py	finetune.ipynb	     query.py
chroma_db	generate_dataset.py  README


In [13]:
import json

with open("deepfake-research-rag/training_data_clean.jsonl") as f:
    data = [json.loads(line) for line in f]

print(f"Loaded {len(data)} examples")
print(data[0])

Loaded 230 examples
{'instruction': 'What type of neural network is used to examine spectral patterns from STFT features?', 'output': 'Convolutional Neural Networks (CNNs) are used to examine spectral patterns from STFT features.'}


In [14]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:42: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [16]:


model = FastLanguageModel.get_peft_model(
    model,
    r =16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [17]:
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}


In [18]:
# convert data into a hugging face dataset
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset = dataset.map(formatting_prompts_func, batched=True)

print(dataset[0]['text'])

Map:   0%|          | 0/230 [00:00<?, ? examples/s]

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What type of neural network is used to examine spectral patterns from STFT features?

### Response:
Convolutional Neural Networks (CNNs) are used to examine spectral patterns from STFT features.<|eot_id|>


In [19]:
#Training

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/230 [00:00<?, ? examples/s]

In [20]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 230 | Num Epochs = 3 | Total steps = 87
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarni

Step,Training Loss
1,3.333520
2,3.289575
3,3.085569
4,3.021295
5,2.871704
6,2.602739
7,2.263924
8,2.375607
9,1.747182
10,1.842885


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Unsloth: Restored 

In [22]:
# test model behavior

FastLanguageModel.for_inference(model)

question = "What features are used to detect audio deepfakes?"

inputs = tokenizer(
    [alpaca_prompt.format(question, "")],
    return_tensors = 'pt'
).to('cuda')

outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
response = tokenizer.batch_decode(outputs)[0]

print(response)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What features are used to detect audio deepfakes?

### Response:
MFCCs, LFCCs, Chroma, Spectral Features, and ZCR are used to detect audio deepfakes.<|eot_id|>


In [23]:
# comparing the model performance before fine tuning

FastLanguageModel.for_inference(model)

with model.disable_adapter():
    inputs = tokenizer(
        [alpaca_prompt.format("What features are used to detect audio deepfakes?", "")],
        return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
    base_response = tokenizer.batch_decode(outputs)[0]

print(base_response)


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What features are used to detect audio deepfakes?

### Response:
To detect audio deepfakes, several features can be used:

1.  **Spectral Features**: These include the Short-Time Fourier Transform (STFT), Mel-Frequency Cepstral Coefficients (MFCCs), and other spectral features that can help identify the source of the audio.
2.  **Mel-Frequency Cepstral Coefficients (MFCCs)**: These are a set of coefficients that describe the spectral characteristics of an audio signal. They can be used to compare the audio of the target and the fake audio and detect any differences.
3.  **Perceptual Features**: These features are based on how human listeners perceive the audio signal. They can include features such as the Pitch, Energy, and Spect


In [24]:
# save new model

model.save_pretrained('lora_adapter')
tokenizer.save_pretrained('lora_adapter')



Unsloth: Restored added_tokens_decoder metadata in lora_adapter/tokenizer_config.json.


('lora_adapter/tokenizer_config.json',
 'lora_adapter/chat_template.jinja',
 'lora_adapter/tokenizer.json')

In [26]:
!du -sh lora_adapter/
!ls -la lora_adapter/

110M	lora_adapter/
total 111912
drwxr-xr-x 2 root root     4096 Sep  6 17:45 .
drwxr-xr-x 1 root root     4096 Sep  6 17:45 ..
-rw-r--r-- 1 root root     1313 Sep  6 17:45 adapter_config.json
-rw------- 1 root root 97307544 Sep  6 17:45 adapter_model.safetensors
-rw-r--r-- 1 root root     3827 Sep  6 17:45 chat_template.jinja
-rw-r--r-- 1 root root     5268 Sep  6 17:45 README.md
-rw-r--r-- 1 root root    50668 Sep  6 17:45 tokenizer_config.json
-rw-r--r-- 1 root root 17209920 Sep  6 17:45 tokenizer.json


In [27]:
!cd deepfake-research-rag && git lfs install
!cd deepfake-research-rag && git lfs track "*.safetensors"
!cd deepfake-research-rag && git add .gitattributes

Updated Git hooks.
Git LFS initialized.
Tracking "*.safetensors"


In [28]:
!cp -r lora_adapter deepfake-research-rag/
!cd deepfake-research-rag && git add lora_adapter/ .gitattributes
!cd deepfake-research-rag && git config user.email "ellohsali@gmail.com"
!cd deepfake-research-rag && git config user.name "Sali Elloh"
!cd deepfake-research-rag && git commit -m "LoRA fine-tuned adapter"


[main 9e5f5b0] LoRA fine-tuned adapter
 7 files changed, 1253433 insertions(+)
 create mode 100644 .gitattributes
 create mode 100644 lora_adapter/README.md
 create mode 100644 lora_adapter/adapter_config.json
 create mode 100644 lora_adapter/adapter_model.safetensors
 create mode 100644 lora_adapter/chat_template.jinja
 create mode 100644 lora_adapter/tokenizer.json
 create mode 100644 lora_adapter/tokenizer_config.json


In [30]:
!cd deepfake-research-rag && git push https://#token@github.com/SaliElloh/deepfake-research-rag.git main

Everything up-to-date
